# Minimum Birkhoff Decomposition

> 🚀 **Skip the local bottleneck.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

Each iteration evaluates parameterized circuits, then a multi-threaded classical `black_box_optimizer` (CPLEX) decodes measured bitstrings into the best convex combination of permutations. As matrix dimensions grow, the number of permutations explodes. QoroService offloads the **circuit evaluations** so the classical optimizer never waits for quantum results.


## Step 0 — Install & Authenticate

```bash
pip install qoro-divi docplex cplex
```


In [ ]:
import json
import os

import numpy as np
import pennylane as qp

from birkhoff import combination_to_integer, run_birkhoff
from divi.backends import MaestroSimulator
from divi.qprog.algorithms._ansatze import GenericLayerAnsatz
from divi.qprog.optimizers import MonteCarloOptimizer, ScipyMethod, ScipyOptimizer


## How It Works

This example finds the **Birkhoff decomposition** of doubly stochastic matrices.  The cost function isn't a Hamiltonian expectation value — it's a CPLEX integer program over measured bitstrings.  That makes it a clean showcase of Divi's flexibility: any problem that fits the shape *"sample circuits, then do classical work on the histograms"* can be wired up the same way.

1. A standalone **`CircuitPipeline`** (`PennyLaneSpecStage → MeasurementStage(COUNTS) → ParameterBindingStage`) maps parameter sets to raw shot histograms.
2. A plain `cost_fn(params)` decodes each bitstring into a permutation combination, runs the multi-threaded **`black_box_optimizer`** (CPLEX) with caching, and returns a scalar loss.
3. A Divi optimizer (`ScipyOptimizer` or `MonteCarloOptimizer`) iterates `cost_fn` to convergence via its public `optimize(...)` API.


## Pick a target matrix


In [ ]:
# Configuration
N_DIM = 3            # Matrix dimension (3 or 4)
K_COMBINATIONS = 2   # Number of permutations in the decomposition
INSTANCE_ID = "1"    # Problem instance (1-10)
MATRIX_TYPE = "sparse"
MAX_ITERATIONS = 10

# Load problem data (resolved relative to the notebook's working directory)
mat_file = f"qbench_0{N_DIM}_{MATRIX_TYPE}.json"
perm_file = f"p{N_DIM}.dat"

with open(mat_file) as f:
    problem_data = json.load(f)
permutations = np.loadtxt(perm_file, dtype=int)

all_permutation_matrices = np.eye(N_DIM, dtype=int)[permutations - 1]

# Parse the specific instance
instance_data = problem_data[INSTANCE_ID]
matrix = np.array(instance_data["scaled_doubly_stochastic_matrix"]).reshape((N_DIM, N_DIM))
scale = instance_data["scale"]
sol_perms = np.eye(N_DIM, dtype=int)[np.array(instance_data["permutations"]).reshape(-1, N_DIM) - 1]
sol_weights = np.array(instance_data["weights"])

print(f"Matrix dimension: {N_DIM}×{N_DIM}")
print(f"Target decomposition: {K_COMBINATIONS} permutations")
print(f"Scale factor: {scale}")
print(f"\nTarget matrix (scaled):")
print(matrix)


---

## Drive the pipeline with COBYLA

Uses Divi's **GenericLayerAnsatz** with RY rotations and CZ entanglers in a brick layout.


In [ ]:
optimizer = ScipyOptimizer(ScipyMethod.COBYLA)
backend = MaestroSimulator(shots=5000)
ansatz = GenericLayerAnsatz([qp.RY], entangler=qp.CZ, entangling_layout="brick")

print("Starting optimization...")
result = run_birkhoff(
    matrix=matrix,
    scale=scale,
    k=K_COMBINATIONS,
    all_perms_matrix=all_permutation_matrices,
    backend=backend,
    optimizer=optimizer,
    max_iterations=MAX_ITERATIONS,
    ansatz=ansatz,
    n_layers=3,
)


## Read the decomposition off the histogram


In [ ]:
if result.combination is not None:
    total_shots = sum(result.final_histogram.values())
    bitstring_width = len(next(iter(result.final_histogram)))
    solution_integer = combination_to_integer(result.combination, K_COMBINATIONS)
    solution_bitstring = format(solution_integer, f"0{bitstring_width}b")
    solution_shots = result.final_histogram.get(solution_bitstring, 0)
    probability = (solution_shots / total_shots) * 100 if total_shots else 0.0

    print("Probability of Best Solution:")
    print(f"  Bitstring '{solution_bitstring}' — {probability:.2f}% ({solution_shots}/{total_shots} shots)")

    print("\nDecomposition (Found):")
    for perm_idx, weight in zip(result.combination, result.weights):
        if weight > 1e-6:
            print(f"  Permutation {perm_idx}: weight = {weight:.6f}")

    reconstructed = sum(
        w * all_permutation_matrices[p]
        for p, w in zip(result.combination, result.weights)
        if w > 1e-6
    )
    original_unscaled = matrix / scale
    error_norm = np.linalg.norm(original_unscaled - reconstructed, ord=2)
    print(f"\nReconstruction error (L2): {error_norm:.6f}")
    print(f"Total circuits: {result.total_circuit_count}")
    print(f"\nOriginal (unscaled):\n{original_unscaled}")
    print(f"\nReconstructed:\n{reconstructed}")
else:
    print("⚠️  No valid decomposition found. Try increasing MAX_ITERATIONS.")


---

## Ready for Larger Matrices?

Try `N_DIM=4` for a harder problem.  When the cost loop gets too heavy for local execution (more iterations, larger `k`, or higher shot counts), switch to QoroService:

```python
from divi.backends import QoroService, JobConfig
qpu_backend = QoroService(job_config=JobConfig(shots=10_000))
```

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.
